In [15]:
# import the required programs - you only need to do this once upon loading the notebook
!pip install cantera

import cantera as ct
from scipy.optimize import root_scalar
import numpy as np

gas = ct.Solution("FFCM2_AME436.yaml")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [16]:
# LH2 - LOX, SSME (RS-25 engine); T_chamber = 3569.5 K, P_star = 114.8 atm, P_exit = 0.17945 atm
h_LH2 = -1.89; MW_LH2 = 0.002; rho_LH2 = 71; DeltaP_LH2 = 7020   # units kcal/mole, kg/mole, kg/m^3, lbf/in^2
h_LO2 = -3.08; MW_LO2 = 0.032; rho_LO2 = 1140; DeltaP_LO2 = 8080  # units kcal/mole, kg/mole, kg/m^3, lbf/in^2
YH2 = 1/7; YO2 = 6/7
h_propellant = (YO2 * h_LO2/MW_LO2 + YH2 * h_LH2/MW_LH2) * 4184
eta_pump = 0.5
Pump_work = (1/eta_pump) * (YO2 * DeltaP_LO2/rho_LO2 + YH2 * DeltaP_LH2/rho_LH2) * 6895  # 6895 N/m^2 = 1 lbf/in^2
h_propellant = h_propellant - Pump_work
P_chamber = 203.7 * 101325                            # chamber pressure = 2997 psi
A_star = (np.pi/4)* 0.260**2                          # throat diameter = 0.260 m
AeOverAstar = 77.5                                    # area ratio Ae/A*
P_amb = 101325                                        # ambient pressure

In [17]:
# compute chamber temperature

def chamber_enthalpy_residual(T):
    gas.TPY = T, P_chamber, mixture
    gas.equilibrate("TP")
    return gas.h - h_propellant

temperature_solution = root_scalar(
    chamber_enthalpy_residual,
    bracket=[3000, 4000],
    method="brentq"
)

T_chamber = temperature_solution.root

gas.TPY = T_chamber, P_chamber, mixture
gas.equilibrate("TP")

h_chamber = gas.h
s_chamber = gas.s

print(f"Chamber temperature = {T_chamber:.2f} K")

Chamber temperature = 3569.51 K


In [18]:
# expand from chamber condition (M << 1) to throat; this is needed to determine mass flow

s_star = s_chamber                                      # isentropic expansion from chamber to throat
P_star = 114.8 * 101325                                 # guess for chamber pressure; adjust until M_star = 1
gas.SP = s_star, P_star                                 # expand isentropically to target throat condition
gas.equilibrate("SP")                                   # this is a rocket - no frozen flow here!
h_star = gas.h                                          # get entalpy, sound speed, and density
c_star = gas.sound_speed
rho_star = gas.density
u_star = np.sqrt(2*(h_chamber - h_star))                # calculate velocity
mdot_star = rho_star * u_star * A_star                  # calculate mass flow - must be same at exit!
M_star = u_star/c_star                                  # calculate Mach number
print(f"Throat Mach number = {M_star:.6g}")             # is M = 1? if not, adjust P_star

Throat Mach number = 0.999642


In [19]:
# expand from throat to exit for a given Ae/A*

s_exit = s_star                                         # isentropic expansion from throat to exit
A_exit = A_star * AeOverAstar                           # exit area
P_exit = 0.17945 * 101325                               # guess for exit pressure; adjust until mdot_exit = mdot_star
gas.SP = s_exit, P_exit                                 # expand isentropically to target throat condition
gas.equilibrate("SP")                                   # this is a rocket - no frozen flow here!
h_exit = gas.h                                          # get entalpy, sound speed, and density
c_exit = gas.sound_speed
rho_exit = gas.density
u_exit = np.sqrt(2*(h_chamber - h_exit))
mdot_exit = rho_exit * A_exit * u_exit                  # calculate exit mass flow - must be same as throat!
mdot_error = (mdot_exit - mdot_star)/(0.5 * (mdot_exit + mdot_star))
print(f"mdot_error = {mdot_error:.8g}")                 # adjust P_exit until mdot_error is <<<< 1
T_exit = gas.T                                          # temperature and Mach number at exit not needed for calculations
M_exit = u_exit/c_exit                                  # ... but nice to know anyway

mdot_error = 7.7971755e-06


In [20]:
# Calculate and print results

Thrust = mdot_exit * u_exit + (P_exit-P_amb) * A_exit   # calculate Thrust
Isp = Thrust/(mdot_exit * 9.806)                        # ... and Isp
print("=== RS-25 Performance ===")
print(f"Chamber Temperature : {T_chamber:.1f} K")
print(f"Throat Pressure     : {P_star / 101325:.3f} atm")
print(f"Throat Mach         : {M_star:.4f}")
print(f"Exit Pressure       : {P_exit / 101325:.5f} atm")
print(f"Exit Temperature    : {T_exit:.1f} K")
print(f"Exit Mach           : {M_exit:.3f}")
print(f"Mass Flow           : {mdot_exit:.2f} kg/s")
print(f"Thrust              : {Thrust / 1000:.2f} kN")
print(f"Specific Impulse    : {Isp:.2f} s")
# Print mole fractions in exhaust
species_names = gas.species_names                       # get all species names and their mole fractions
mole_fractions = gas.X
# Combine names & mole fractions (name, value) then sort descending order of mole fraction
species_data = sorted(zip(species_names, mole_fractions), key=lambda x: x[1], reverse=True)
print()
print(f"{'Species':<15} | {'Mole Fraction':<15}")
print("-" * 35)
for name, fraction in species_data[:10]:                 # print only 10 most abundant species
    if fraction > 1e-9:                                  # ... and only print species with significant concentrations
        print(f"{name:<15} | {fraction:.6e}")

=== RS-25 Performance ===
Chamber Temperature : 3569.5 K
Throat Pressure     : 114.800 atm
Throat Mach         : 0.9996
Exit Pressure       : 0.17945 atm
Exit Temperature    : 1169.7 K
Exit Mach           : 4.691
Mass Flow           : 474.24 kg/s
Thrust              : 1729.53 kN
Specific Impulse    : 371.91 s

Species         | Mole Fraction  
-----------------------------------
H2O             | 7.560472e-01
H2              | 2.439527e-01
H               | 1.287891e-07
OH              | 3.331651e-09
